This notebook presents a production-oriented fraud detection approach focused on maximizing monetary fraud recall while keeping customer friction low.
Instead of optimizing a single metric, we define a simple decision policy based on transaction amount and validate it on a held-out test set.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

## Introduction

Fraud detection is a highly imbalanced classification problem where traditional metrics such as accuracy can be misleading.
In real-world payment systems, missing a large fraudulent transaction is usually far more costly than missing several small ones.

This notebook presents a production-oriented fraud detection approach focused on maximizing **fraud recall by monetary value**,
while keeping customer friction low. Instead of optimizing a single threshold globally, we define a simple and interpretable
decision policy based on transaction amount.

The goal is not to win a leaderboard, but to demonstrate how a machine learning model can be transformed into a practical
decision system suitable for real-world deployment.

## Data Overview

The dataset consists of two files:

- `fraudTrain.csv`: used exclusively for model training and rule definition
- `fraudTest.csv`: a held-out test set used only for final evaluation

Both datasets share the same schema and come from the same data-generating process.
No cross-validation is performed in order to preserve a clear and realistic train / test separation.

## Exploratory Clustering Analysis

Before training any supervised model, an unsupervised clustering analysis was performed using K-means.

The objective was not to detect fraud directly, but to explore whether transactions naturally group into
distinct behavioral patterns and whether those groups exhibit different fraud rates.

The clustering was based on a reduced set of numerical features, including transaction amount,
temporal features, and distance-related variables.

The analysis showed that:
- Some clusters exhibited significantly higher fraud ratios than others
- Transaction amount and temporal patterns were key drivers of separation
- Geographic raw coordinates provided limited additional value once distance-based features were included

This exploratory step helped confirm that fraud risk is not uniformly distributed and supported
the decision to focus the supervised model on stable, low-dimensional features.

## Baseline Model and Feature Diagnostics

Before training the final gradient boosting model, a logistic regression baseline was used as a diagnostic tool.
The objective was not to maximize performance, but to evaluate feature relevance, statistical significance,
and multicollinearity.

The logistic model provided:
- Coefficient estimates with statistical significance
- Variance Inflation Factor (VIF) analysis to detect collinearity
- A transparent baseline for comparison with more complex models

This analysis confirmed that transaction amount, temporal features, and basic customer attributes
contributed meaningful signal, while several other variables showed limited or unstable effects.
Highly collinear features were removed to improve robustness.

The insights from this step informed the final feature selection used in the LightGBM model.

### Logistic Regression Diagnostics

As part of the feature selection process, a logistic regression model was fitted on the training set.
The objective was not to optimize predictive performance, but to assess feature relevance,
statistical significance, and potential multicollinearity.

Both temporal representations — the full day-of-week encoding and a binary weekend indicator —
were evaluated jointly to understand their complementary contributions.

The table below summarizes the main effects identified by the logistic regression:

| Variable                     | Effect sign | Statistical significance | Interpretation |
|------------------------------|-------------|--------------------------|----------------|
| Transaction amount (`amt`)   | Positive    | Strong (p < 0.001)       | Higher amounts are associated with higher fraud risk |
| Age                          | Positive    | Strong (p < 0.001)       | Fraud risk increases with customer age |
| Gender                       | Positive    | Strong (p < 0.001)       | Moderate but consistent effect |
| Day of week (`dayofweek`)    | Positive    | Moderate (p < 0.05)      | Systematic variation across weekdays |
| Weekend indicator (`is_weekend`) | Positive| Moderate (p < 0.01)  | Additional behavioral shift on weekends |
| Hour (cyclical encoding)     | Positive    | Strong (p < 0.001)       | Strong time-of-day dependency |

Variance Inflation Factor (VIF) analysis indicated moderate collinearity between `dayofweek`
and `is_weekend`, as expected. However, both variables were retained at this stage because they
capture different temporal effects: gradual weekly patterns and discrete weekend behavior.

This diagnostic step confirmed that temporal structure is an important driver of fraud risk
and informed the final feature set used in the gradient boosting model, without directly
determining the final decision thresholds.


## Feature Selection

The initial dataset contains personal identifiers, raw timestamps, and geographic coordinates.
For a production-oriented approach, features were selected according to three criteria:

- Stability across time
- Low risk of data leakage
- Interpretability for decision-making

The final model uses a small set of aggregated and transformed features:

- **Transaction amount (`amt`)**
- **Customer age**
- **Customer gender**
- **Day of week**
- **Weekend indicator**
- **Cyclical encoding of transaction hour**

Raw identifiers, exact locations, and high-cardinality categorical variables were deliberately excluded
to reduce overfitting and improve generalization.

This feature set favors robustness and interpretability over maximum predictive power.

## Model Training

A LightGBM binary classifier was trained using the training dataset only.
Class imbalance was handled through class weighting rather than resampling.

No extensive hyperparameter tuning was performed.
The focus was on obtaining a stable and well-regularized model rather than maximizing leaderboard metrics.

Once trained, the model was frozen and used only for scoring the test set.

## Evaluation on Test Set

All results reported in this section correspond to the held-out test set.
No model parameters, thresholds, or amount cutoffs were adjusted after training.

### Global Performance

The table below summarizes the overall system performance:

| Metric                         | Value  |
|--------------------------------|--------|
| Fraud recall (transactions)    | 85.6 % |
| Fraud recall (monetary value)  | 91.5 % |
| Precision                      | 8.6 %  |
| Transactions flagged           | 3.83 % |
| Monetary volume flagged        | 4.93 % |

With less than 4% of transactions flagged, the system captures over 90% of the total fraudulent monetary value.

---

### Performance by Transaction Amount

To better understand system behavior, results are broken down by transaction amount relative to the training mean.

#### Transactions Below Mean Amount

| Metric                         | Value  |
|--------------------------------|--------|
| Fraud recall (transactions)    | 51.8 % |
| Fraud recall (monetary value)  | 54.0 % |
| Precision                      | 2.6 %  |
| Transactions flagged           | 3.92 % |
| Monetary volume flagged        | 4.73 % |

Lower-value fraud is more diffuse and harder to separate, resulting in lower recall and precision.
This risk is accepted to avoid excessive customer friction.

---

#### Transactions Above Mean Amount

| Metric                         | Value  |
|--------------------------------|--------|
| Fraud recall (transactions)    | 94.6 % |
| Fraud recall (monetary value)  | 95.6 % |
| Precision                      | 12.9 % |
| Transactions flagged           | 3.77 % |
| Monetary volume flagged        | 5.00 % |

High-value fraudulent transactions are detected with very high recall while keeping intervention rates low.
This confirms the effectiveness of applying a more permissive threshold to higher-risk, higher-impact transactions.

---

### Interpretation

The results demonstrate that a simple amount-aware decision policy can substantially improve economic outcomes
without increasing overall system complexity.

Rather than attempting to maximize fraud detection uniformly, the system prioritizes the detection of
high-impact fraudulent transactions, achieving a strong balance between risk r

## Conclusion

This project demonstrates that effective fraud detection is not only a modeling problem, but a decision-making problem.

By combining a well-regularized LightGBM model with a simple, amount-aware decision policy, the system achieves
strong economic performance while keeping customer friction low.
Rather than maximizing a single metric, the approach prioritizes the detection of high-impact fraudulent transactions.

With less than 4% of transactions flagged, the system captures over 90% of the total fraudulent monetary value
on a held-out test set, making it suitable for real-world deployment.

The key takeaway is that simple, interpretable rules built on top of a robust model can often outperform
more complex strategies when evaluated through a business-oriented lens.